In [1]:
import sys
sys.path.append("../")

from models.bulge_models import BulgeTemplates
from models.templates import LorimerDiskTemplate
from models.np_model import NPModel

%load_ext autoreload
%autoreload 2

In [2]:
import jax
import jax.numpy as jnp
import numpyro.distributions as dist

In [3]:
import healpy as hp
import matplotlib.pyplot as plt

In [25]:
# bulge_template_names = ["mcdermott2022", "mcdermott2022_bbp", "mcdermott2022_x", "macias2019", "coleman2019"]
# bulge_templates = jnp.array([BulgeTemplates(template_name=template_name)() for template_name in bulge_template_names])

# # Individually normalize the bulge templates
# bulge_templates = bulge_templates / jnp.mean(bulge_templates[:, :], axis=-1)[:, None]

# n_templates = len(bulge_templates)
# proposal = dist.Dirichlet(jnp.ones((n_templates,)) / n_templates)
# thetas = proposal.sample(key=jax.random.PRNGKey(32))

# temp_bulge = jnp.sum(thetas[:, None] * bulge_templates, 0)

# hp.cartview(temp_bulge, lonra=[-20,20], latra=[-20,20])

In [26]:
r_outer = 25
l_max = 0

vary_disk = True
vary_gamma = True
bulge_hybrid = True

dif = "ModelO"
ps_cat = "3fgl"
nside = 128

npmodel = NPModel(dif=dif, r_outer=r_outer, l_max=l_max, vary_disk=vary_disk, vary_gamma=vary_gamma, bulge_hybrid=bulge_hybrid, ps_cat=ps_cat, nside=nside)
svi_results = npmodel.fit_svi(rng_key=jax.random.PRNGKey(4234), n_steps=5000)


Loading the psf correction from: /net/rcstorenfs02/ifs/rc_labs/dvorkin_lab/smsharma/mi-attribution/notebooks/psf_dir/Fermi_PSF_2GeV2_nside128.npy
Max photon count is 103


100%|██████████| 5000/5000 [05:18<00:00, 15.70it/s, init loss: 29583.1776, avg. loss [4751-5000]: 20102.0690]


In [27]:
import arviz as az

posterior = npmodel.get_posterior_samples(rng_key=jax.random.PRNGKey(42342), num_samples=50000)

In [28]:
posterior_bulge = {}
for key in list(posterior.keys()):
    if "theta" in key:
        posterior_bulge[key] = posterior[key]
        posterior.pop(key, None)

In [29]:
az.summary(posterior)

Shape validation failed: input_shape: (1, 50000), minimum_shape: (chains=2, draws=4)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
C,2.840,0.870,1.375,4.519,0.004,0.003,49877.0,49501.0,NaN
S_bub,1.110,0.108,0.909,1.311,0.000,0.000,49773.0,49624.0,NaN
S_dif,11.258,0.175,10.940,11.594,0.001,0.001,48909.0,49794.0,NaN
S_gce,1.203,0.268,0.710,1.688,0.001,0.001,49732.0,49378.0,NaN
S_ics,4.734,0.341,4.164,4.998,0.002,0.001,50002.0,48525.0,NaN
S_iso,0.932,0.274,0.453,1.449,0.001,0.001,48627.0,48766.0,NaN
S_psc,2.380,1.374,0.129,4.601,0.006,0.004,50024.0,49745.0,NaN
Sps_dsk,1.721,0.244,1.258,1.994,0.001,0.001,50433.0,48954.0,NaN
Sps_gce,0.379,0.183,0.088,0.722,0.001,0.001,48899.0,49654.0,NaN
f_bulge_poiss,0.414,0.264,0.012,0.866,0.001,0.001,48198.0,49907.0,NaN


In [32]:
jnp.mean(posterior_bulge['theta_bulge_poiss'], 0)

DeviceArray([0.05030361, 0.04859829, 0.09764139, 0.36232287, 0.44113384],            dtype=float64)